# Bounded T4 beam-search comparison
CPU preparation is implemented by `prepare_decoding_probe.py`. It verifies the original 180k model, all test-question positions, model loading and a short four-beam generation before saving `source.tar`, `ready.json` and `launch.py`. Upload that prepared folder to `MyDrive/HSE_NLP_HW2/prepared/beam4-180k-v1`. The original `prepared/inference-speed-180k-v1/inputs.tar` must already exist.
Select one T4 and run only the next cell. Two full 1,919-question FP32 passes use four beams and batch 128. The child has a 600-second bound. Exact outputs and provenance are saved to Drive before export. Verify actual disconnection after the completion record appears; manually release if Colab stays connected. No automatic bot submission.


In [ ]:
from google.colab import drive, runtime
try:
    import torch
    assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name()
    drive.mount('/content/drive', timeout_ms=90000)
    import os, sys, json, hashlib, tarfile, subprocess, asyncio, signal, time
    from pathlib import Path
    root=Path('/content/drive/MyDrive/HSE_NLP_HW2')
    prepared=root/'prepared/beam4-180k-v1'
    ready=json.loads((prepared/'ready.json').read_text())
    assert ready['cpu_verified']
    work=Path('/content/beam4-source'); work.mkdir(exist_ok=False)
    source=(prepared/'source.tar').read_bytes()
    assert hashlib.sha256(source).hexdigest()==ready['source_sha256']
    (work/'source.tar').write_bytes(source)
    with tarfile.open(work/'source.tar') as archive: archive.extractall(work,filter='data')
    assert (work/'source_commit.txt').read_text().strip()==ready['source_commit']
    inputs=(root/'prepared/inference-speed-180k-v1/inputs.tar').read_bytes()
    assert hashlib.sha256(inputs).hexdigest()==ready['inputs_sha256']
    (work/'inputs.tar').write_bytes(inputs)
    with tarfile.open(work/'inputs.tar') as archive: archive.extractall(work/'inputs',filter='data')
    subprocess.run([sys.executable,'-m','pip','-q','install','transformers==4.57.6','pyarrow>=18'],check=True,timeout=180)
    os.environ['USE_TF']='0'; os.environ['USE_FLAX']='0'
    decoding_log=(prepared/'execution.log').open('w')
    command=[sys.executable,'-u','-m','homeworks.hw2.decoding_t4',
        '--model-dir',str(work/'inputs/model'),'--cache-dir',str(work/'inputs/cache'),
        '--output-dir',str(root/'benchmarks/beam4-180k-v1'),
        '--expected-model-sha256',ready['model_sha256'],'--beams','4','--batch-size','128']
    decoding_process=subprocess.Popen(command,cwd=work,stdout=decoding_log,stderr=subprocess.STDOUT,start_new_session=True)
    async def finish_decoding():
        status={'pid':decoding_process.pid,'started_utc':time.time()}
        try:
            await asyncio.wait_for(asyncio.to_thread(decoding_process.wait),timeout=600)
            status['returncode']=decoding_process.returncode
        except BaseException as error:
            status['error']=type(error).__name__+': '+str(error)
            if decoding_process.poll() is None:
                os.killpg(decoding_process.pid,signal.SIGTERM)
                try: await asyncio.wait_for(asyncio.to_thread(decoding_process.wait),timeout=10)
                except asyncio.TimeoutError: os.killpg(decoding_process.pid,signal.SIGKILL)
        finally:
            decoding_log.close(); status['finished_utc']=time.time()
            try: (prepared/'execution.complete.json').write_text(json.dumps(status,indent=2))
            finally: runtime.unassign()
    decoding_task=asyncio.create_task(finish_decoding())
    print('T4_DECODING_STARTED',decoding_process.pid,'600-second bound; verify runtime release after completion.')
except BaseException:
    runtime.unassign()
    raise
